# B06B — Train-calibrated feature KD (adaptasi terinspirasi SSID-KD)
Bukan reproduksi SSID-KD asli. GAP Teacher beku → projector Student → standardisasi per citra → KL distribusi kanal. Loss = weighted BCE + lambda × T² KL(Teacher || Student). Tidak menggunakan Hinton logit loss.

Enam run per dataset: dua supervised controls baru + empat KD. Gunakan seed42, full512, batch8, 5 head +25 finetune, tanpa early stopping. Pilih checkpoint berdasarkan validation macro-F1 lalu recall; head tetap eligible dan menang bila tie. Teacher B04B tidak dilatih ulang. Student mulai dari ImageNet, bukan checkpoint Hinton.

**Kalibrasi:** maksimal8 minibatch train unik, batch8, sampling acak deterministik pada prevalensi alami (bukan oversampling), sebelum head training dan tanpa update bobot. Ukur kedua gradien pada backbone yang sama; dropout dimatikan hanya saat kalibrasi. lambda=clip(0.5 × median(norm(supervised)/norm(feature)),0.01,10). Target0.5 dan clipping adalah heuristik pradefinisi, bukan nilai paper/optimal. Lambda tetap saat head dan finetune; rasio dapat berubah selama training. Catat cosine, rentang dan median.

**Resume:** checkpoint model, projector, optimizer, epoch, best dan history setiap akhir epoch. Epoch yang terputus diulang. Shuffle, flip dan dropout stateless; checkpoint disimpan per fase. Jangan mengubah konfigurasi untuk resume. Output baru terpisah dari B04/B05/B06A.

Trainer memakai dropout stateless untuk resume; berbeda detail RNG dari B05. Dua control baru membantu mengaudit pengaruh perubahan trainer. Perbandingan dengan Hinton tetap mencatat perubahan RNG dan biaya kalibrasi/projector; jangan mengatribusikan semua selisih hanya pada loss.

**Test selalu nonaktif.** Jalankan Cell1–7 di Colab GPU. Cell6 menjalankan kalibrasi dan training; Cell7 dapat dijalankan untuk laporan parsial jika ada error. Sumber inspirasi: https://github.com/cuiyixin555/SSID-KD ; implementasi asli deraining tidak direproduksi di sini.

## Cell 1 — Runtime

In [ ]:
import os, sys, subprocess, importlib.metadata as metadata
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
import tensorflow as tf
minor='.'.join(tf.__version__.split('.')[:2])
try: installed=metadata.version('tf-keras')
except metadata.PackageNotFoundError: installed=''
if '.'.join(installed.split('.')[:2])!=minor:
    if 'tf_keras' in sys.modules:raise RuntimeError('Restart runtime sebelum mengubah tf-keras')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'tf-keras~={minor}.0'])
import tf_keras as keras
import numpy as np, pandas as pd
import json, hashlib, zipfile, gc, traceback, ast, linecache
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
from google.colab import drive, files
from IPython.display import display
gpus=tf.config.list_physical_devices('GPU')
if not gpus:raise RuntimeError('Pilih runtime GPU di Colab')
for gpu in gpus:tf.config.experimental.set_memory_growth(gpu,True)
keras.mixed_precision.set_global_policy('float32')
tf.config.experimental.enable_op_determinism()
drive.mount('/content/drive')
print(tf.__version__,keras.__version__,gpus)

## Cell 2 — Konfigurasi

In [ ]:
DATASET='cbis_ddsm'  # cbis_ddsm | cmmd | inbreast | mias
SEED=42
DOWNLOAD_REPORT=True
RESEARCH=Path('/content/drive/MyDrive/Research')
SOURCE_ROOT=RESEARCH/'outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1'
IDS={'cbis_ddsm':'7a4bf6d1e3f6','cmmd':'f81ede1d807a','inbreast':'3f2fbdc2b913','mias':'672f572d6c33'}
REGISTRY_PATH=RESEARCH/'outputs/benchmark_4datasets/b04_preprocessing_v1/1dcb7ddadfe6/registry.json'
assert DATASET in IDS
assert SEED==42
SOURCE=SOURCE_ROOT/DATASET/IDS[DATASET]
OUT=RESEARCH/'outputs/benchmark_4datasets/b06b_calibrated_feature_kd_v1'/DATASET
OUT.mkdir(parents=True,exist_ok=True)
def read_json(p):return json.loads(Path(p).read_text())
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
def save(p,v):
    p=Path(p);tmp=p.with_suffix(p.suffix+'.tmp');tmp.write_text(json.dumps(v,indent=2,default=str));os.replace(tmp,p)
def guarded(p,v):
    if Path(p).exists() and read_json(p)!=v:raise RuntimeError('Frozen configuration changed: '+str(p))
    save(p,v)
CONFIG=dict(version='b06b_v1',dataset=DATASET,seed=42,batch=8,head_epochs=5,finetune_epochs=25,
    head_lr=.001,finetune_lr=.00001,temperature=2.,normalization='standardize',epsilon=1e-5,
    calibration_batches=8,target_ratio=.5,lambda_min=.01,lambda_max=10.,
    calibration='before head, natural train sample, no dropout, shared backbone gradients',
    threshold=.5,dropout=.2,rng='stateless per epoch/batch',test=False,
    selection='validation macro_f1 then recall; earliest/head on tie',
    tensorflow=tf.__version__,tf_keras=keras.__version__)
ERRORS=[];SOURCE_HASHES={}


## Cell 3 — Provenance, train/validation dan helper

In [ ]:
proto=read_json(SOURCE/'protocol.json')
if hashlib.sha256(json.dumps(proto,sort_keys=True).encode()).hexdigest()[:12]!=IDS[DATASET]:
    raise RuntimeError('ID protokol B05B tidak cocok')
helper_path=SOURCE/'b04_compat.py'
expected='2822801a4f73ab90d6df8215d12fad03ea832cbc2300bdeb36d729d66ab66297'
if sha(helper_path)!=expected or proto['b04_compat_sha256']!=expected:raise RuntimeError('Helper berubah')
# Muat hanya fungsi model/inferensi yang dibutuhkan. Tidak memanggil loader test atau training.
source_text=helper_path.read_text(); tree=ast.parse(source_text)
keep={'REGISTRY','json_safe','save_json','file_sha','metrics','make_dataset','build_model','weight_snapshot'}
nodes=[n for n in tree.body if isinstance(n,(ast.Import,ast.ImportFrom)) or
       (isinstance(n,ast.FunctionDef) and n.name in keep) or
       (isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='REGISTRY' for t in n.targets))]
helper_code=ast.unparse(ast.Module(body=nodes,type_ignores=[]))
filename='/content/b06a_inference_helpers.py'
Path(filename).write_text(helper_code)
linecache.cache[filename]=(len(helper_code),None,helper_code.splitlines(True),filename)
exec(compile(helper_code,filename,'exec'),globals())
(OUT/'inference_helpers.py').write_text(helper_code)
entry=read_json(REGISTRY_PATH)[DATASET]['full_512']
if entry['status']!='READY':raise RuntimeError('Cache belum READY')
folder=Path(entry['path']);meta=read_json(folder/'READY.json')
if meta['dataset']!=DATASET or meta['image_size']!=512:raise RuntimeError('Cache salah')
if meta['file_sha256']!=proto['data_sha256'] or meta['cohort_sha256']!=proto['cohort_sha256']:
    raise RuntimeError('Data berbeda dari B05B')
FRAMES={}
for split in ['train','validation']:
    path=folder/(split+'.csv')
    if sha(path)!=meta['file_sha256'][path.name]:raise RuntimeError('CSV berubah')
    f=pd.read_csv(path,dtype={'sample_id':str,'patient_id':str})
    if f[['sample_id','patient_id','label']].isna().any().any():raise RuntimeError('Missing ID/label')
    if f.sample_id.duplicated().any() or not f.label.isin([0,1]).all():raise RuntimeError('ID/label invalid')
    if set(f.split)!={split} or set(f.label)!={0,1}:raise RuntimeError('Split/kelas invalid')
    FRAMES[split]=f
if set(FRAMES['train'].patient_id)&set(FRAMES['validation'].patient_id):raise RuntimeError('Patient leakage')
if set(FRAMES['train'].sample_id)&set(FRAMES['validation'].sample_id):raise RuntimeError('Image leakage')
needed=pd.concat([FRAMES['train'],FRAMES['validation']],ignore_index=True)
IMAGE_DIR=Path('/content/b06a_images')/DATASET/meta['file_sha256']['images.zip'][:16]
IMAGE_DIR.mkdir(parents=True,exist_ok=True)
# Opaque archive hash includes the complete archive; no test member is opened.
if sha(folder/'images.zip')!=meta['file_sha256']['images.zip']:raise RuntimeError('Archive berubah')
with zipfile.ZipFile(folder/'images.zip') as archive:
    for row in needed.itertuples():
        member=row.cache_member
        if Path(member).name!=member:raise RuntimeError('Unsafe cache member')
        dest=IMAGE_DIR/member
        if not dest.exists():dest.write_bytes(archive.read(member))
        with Image.open(dest) as im:a=np.asarray(im)
        if a.shape!=(512,512) or hashlib.sha256(a.tobytes()).hexdigest()!=row.processed_pixel_sha256:
            raise RuntimeError('Pixel cache mismatch')
counts=FRAMES['train'].label.value_counts()
CLASS_WEIGHTS=tf.constant([len(FRAMES['train'])/(2*counts[i]) for i in [0,1]],tf.float32)
SOURCES={}
for teacher in ['vgg16','densenet121']:
    src=proto['teacher_sources'][teacher];p=Path(src['folder'])
    if sha(p/'config.json')!=src['config_sha256']:raise RuntimeError('Teacher config berubah')
    done=read_json(p/'COMPLETE.json')
    if done['best_checkpoint']!=src['checkpoint']:raise RuntimeError('Teacher checkpoint berubah')
    for name,digest in src['checkpoint_sha256'].items():
        part=Path(src['checkpoint']).parent/name
        if sha(part)!=digest:raise RuntimeError('Teacher weights berubah')
        SOURCE_HASHES[str(part)]=digest
    SOURCE_HASHES[str(p/'config.json')]=sha(p/'config.json')
    SOURCES[teacher]=src
save(OUT/'source_reference.json',dict(protocol=proto,teacher_sources=SOURCES,hashes=SOURCE_HASHES))
print('Train:',len(FRAMES['train']),'Validation:',len(FRAMES['validation']))
print('Class weights:',CLASS_WEIGHTS.numpy(),'| Test tidak dibuka')

CONFIG['source_protocol_sha256']=sha(SOURCE/'protocol.json')
CONFIG['data_sha256']=meta['file_sha256']
CONFIG['teacher_sources']=SOURCES
print('Data ready. Test CSV tidak dibaca.')

## Cell 4 — Cache Teacher dan kalibrasi

In [ ]:
ENGINE_SOURCE="import time\nfrom sklearn.metrics import ConfusionMatrixDisplay\nimport matplotlib.pyplot as plt\n\ndef prep(x,key):return getattr(keras.applications,REGISTRY[key][2]).preprocess_input(tf.identity(x))\ndef feature_model(m):return keras.Model(m.input,m.get_layer('head_dropout').input)\ndef normalize(h):\n    mean,var=tf.nn.moments(tf.cast(h,tf.float32),axes=[-1],keepdims=True)\n    return (h-mean)*tf.math.rsqrt(var+1e-5)\ndef fkl(ht,hp):\n    lt=tf.nn.log_softmax(tf.stop_gradient(normalize(ht))/2.,axis=-1)\n    ls=tf.nn.log_softmax(normalize(hp)/2.,axis=-1)\n    return 4.*tf.reduce_mean(tf.reduce_sum(tf.exp(lt)*(lt-ls),axis=-1))\ndef sup(y,z):\n    y=tf.cast(tf.reshape(y,[-1,1]),tf.float32)\n    w=tf.gather(CLASS_WEIGHTS,tf.cast(tf.reshape(y,[-1]),tf.int32))\n    return tf.reduce_mean(w*tf.reshape(tf.nn.sigmoid_cross_entropy_with_logits(labels=y,logits=z),[-1]))\ndef weights_hash(m):\n    h=hashlib.sha256()\n    for w in m.weights:h.update(w.numpy().tobytes())\n    return h.hexdigest()\ndef raw_ds(frame):\n    paths=[str(IMAGE_DIR/n) for n in frame.cache_member]\n    d=tf.data.Dataset.from_tensor_slices(paths)\n    def load(p):return tf.ensure_shape(tf.cast(tf.image.decode_png(tf.io.read_file(p),channels=3),tf.float32),[512,512,3])\n    return d.map(load,num_parallel_calls=2,deterministic=True).batch(8).prefetch(1)\ndef raw_at(frame):\n    return tf.constant(np.stack([np.asarray(Image.open(IMAGE_DIR/n).convert('RGB'),dtype=np.float32) for n in frame.cache_member]))\ndef new_student(student,dim,stage,weights):\n    m=build_model(student,512,42,weights=weights,stage=stage)\n    fm=feature_model(m)\n    pj=keras.layers.Dense(dim,kernel_initializer=keras.initializers.GlorotUniform(seed=142),name='feature_projector')\n    pj(tf.zeros([1,int(fm.output_shape[-1])]))\n    return m,fm,pj\n\ndef teacher_cache(teacher):\n    src=SOURCES[teacher];out=OUT/'teacher_cache'/teacher;out.mkdir(parents=True,exist_ok=True)\n    manifest=dict(source=src,data=meta['file_sha256'],train_ids=FRAMES['train'].sample_id.tolist(),batch=8,views=['original','horizontal_flip'])\n    guarded(out/'manifest.json',manifest)\n    if (out/'COMPLETE.json').exists():\n        done=read_json(out/'COMPLETE.json')\n        for name,h in done['hashes'].items():\n            if sha(out/name)!=h:raise RuntimeError('Cache changed')\n        return np.load(out/'features.npy',mmap_mode='r')\n    start=time.perf_counter();keras.backend.clear_session();gc.collect()\n    m=build_model(teacher,512,42,weights=None,stage='head')\n    weight_snapshot(m).restore(src['checkpoint']).assert_existing_objects_matched();m.trainable=False\n    before=weights_hash(m)\n    ref=FRAMES['validation'];saved=pd.read_csv(Path(src['folder'])/'validation_predictions.csv',dtype={'sample_id':str,'patient_id':str},float_precision='round_trip')\n    if saved.sample_id.duplicated().any() or set(saved.sample_id)!=set(ref.sample_id):raise RuntimeError('Teacher IDs differ')\n    saved=saved.set_index('sample_id').loc[ref.sample_id]\n    for c in ['label','patient_id']:\n        if not np.array_equal(saved[c].to_numpy(),ref[c].to_numpy()):raise RuntimeError('Teacher labels/IDs differ')\n    p=m.predict(make_dataset(ref,IMAGE_DIR,teacher,8,42,size=512),verbose=0).reshape(-1);q=saved.probability.to_numpy()\n    if not np.allclose(p,q,atol=1e-5,rtol=1e-5) or not np.array_equal(p>=.5,q>=.5):raise RuntimeError('Teacher restore failed')\n    save(out/'restore_audit.json',dict(passed=True,max_abs_difference=float(np.max(abs(p-q)))))\n    fm=feature_model(m);views=[]\n    for flip in [False,True]:\n        chunks=[]\n        for x in raw_ds(FRAMES['train']):\n            if flip:x=tf.reverse(x,[2])\n            a=fm(prep(x,teacher),training=False).numpy()\n            if not np.isfinite(a).all():raise RuntimeError('Nonfinite teacher features')\n            chunks.append(a)\n        views.append(np.concatenate(chunks))\n    features=np.stack(views,axis=1).astype('float32')\n    if before!=weights_hash(m):raise RuntimeError('Teacher changed')\n    np.save(out/'features.npy',features)\n    save(out/'COMPLETE.json',dict(minutes=(time.perf_counter()-start)/60,teacher_unchanged=True,\n        shape=list(features.shape),hashes={n:sha(out/n) for n in ['features.npy','restore_audit.json','manifest.json']}))\n    del m,fm;keras.backend.clear_session();gc.collect()\n    return np.load(out/'features.npy',mmap_mode='r')\n\ndef calibrate(student,teacher,features,out):\n    path=out/'calibration.json'\n    if path.exists():\n        c=read_json(path)\n        if sha(out/'calibration_batches.csv')!=c['batches_sha256']:raise RuntimeError('Calibration CSV changed')\n        return c\n    start=time.perf_counter();keras.backend.clear_session();gc.collect()\n    m,fm,pj=new_student(student,features.shape[-1],'finetune','imagenet');initial=weights_hash(m)\n    base=next(l for l in m.layers if isinstance(l,keras.Model));vs=base.trainable_variables\n    ids=np.random.default_rng(42).permutation(len(FRAMES['train']))[:min(64,(len(FRAMES['train'])//8)*8)]\n    if len(ids)<8:raise RuntimeError('Need at least 8 train images')\n    rows=[]\n    for k in range(0,len(ids),8):\n        ix=ids[k:k+8];f=FRAMES['train'].iloc[ix];x=raw_at(f);ht=tf.constant(features[ix,0,:])\n        with tf.GradientTape(persistent=True) as tape:\n            hs=fm(prep(x,student),training=False);cl=m.get_layer('probability')\n            ls=sup(f.label.to_numpy(),tf.matmul(hs,cl.kernel)+cl.bias);lk=fkl(ht,pj(hs))\n        a=tape.gradient(ls,vs);b=tape.gradient(lk,vs);del tape\n        if any(g is None for g in a+b):raise RuntimeError('Disconnected backbone gradient')\n        for g in a+b:tf.debugging.assert_all_finite(g,'Calibration gradient')\n        na=float(tf.linalg.global_norm(a));nb=float(tf.linalg.global_norm(b))\n        if min(na,nb)<=1e-12:raise RuntimeError('Degenerate gradient; inspect before calibration')\n        dot=float(tf.add_n([tf.reduce_sum(u*v) for u,v in zip(a,b)]))\n        rows.append(dict(batch=k//8,images=len(ix),positives=int(f.label.sum()),sup_norm=na,feature_norm=nb,\n            ratio_sup_feature=na/nb,cosine=dot/(na*nb),sup_loss=float(ls),feature_loss=float(lk)))\n    if initial!=weights_hash(m):raise RuntimeError('Calibration updated Student')\n    frame=pd.DataFrame(rows);frame.to_csv(out/'calibration_batches.csv',index=False)\n    raw=.5*float(frame.ratio_sup_feature.median());lam=float(np.clip(raw,.01,10.))\n    c=dict(lambda_feature=lam,unclipped_lambda=raw,clipped=bool(lam!=raw),initial_student_sha256=initial,\n        batches=len(rows),sample_ids=FRAMES['train'].iloc[ids].sample_id.tolist(),minutes=(time.perf_counter()-start)/60,\n        cosine_median=float(frame.cosine.median()),cosine_min=float(frame.cosine.min()),cosine_max=float(frame.cosine.max()),\n        ratio_min=float(frame.ratio_sup_feature.min()),ratio_median=float(frame.ratio_sup_feature.median()),ratio_max=float(frame.ratio_sup_feature.max()),\n        achieved_median_weighted_ratio=float(np.median(lam*frame.feature_norm/frame.sup_norm)),\n        batches_sha256=sha(out/'calibration_batches.csv'))\n    save(path,c);del m,fm,pj;keras.backend.clear_session();gc.collect();return c\n\ndef evaluate(m,student,folder,extra):\n    f=FRAMES['validation'];p=m.predict(make_dataset(f,IMAGE_DIR,student,8,42,size=512),verbose=0).reshape(-1)\n    result=dict(metrics(f.label,p),**extra)\n    table=f[['sample_id','patient_id','label']].copy();table['probability']=p;table['prediction']=(p>=.5).astype(int)\n    table.to_csv(folder/'validation_predictions.csv',index=False);save(folder/'validation_metrics.json',result)\n    return result\n\ndef phase(student,teacher,features,lam,out,stage,head_checkpoint=None):\n    out.mkdir(parents=True,exist_ok=True);donepath=out/'COMPLETE.json'\n    if donepath.exists():return read_json(donepath)\n    keras.backend.clear_session();gc.collect();ishead=stage=='head';budget=5 if ishead else 25\n    latest=tf.train.latest_checkpoint(str(out/'last'))\n    dim=features.shape[-1] if features is not None else 1\n    m,fm,pj=new_student(student,dim,stage,'imagenet' if ishead and not latest else None)\n    snap=tf.train.Checkpoint(backbone=next(l for l in m.layers if isinstance(l,keras.Model)),classifier=m.get_layer('probability'),projector=pj)\n    if not latest and not ishead:snap.restore(head_checkpoint).assert_existing_objects_matched()\n    if ishead and not latest:save(out/'initialization.json',dict(student_sha256=weights_hash(m)))\n    variables=m.trainable_variables+(pj.trainable_variables if teacher!='none' else [])\n    opt=keras.optimizers.Adam(.001 if ishead else .00001);opt.build(variables)\n    ep=tf.Variable(0,dtype=tf.int64,trainable=False)\n    state=tf.Variable(json.dumps(dict(history=[],best_score=[-1.,-1.],best_checkpoint=None)),dtype=tf.string,trainable=False)\n    ck=tf.train.Checkpoint(model=m,projector=pj,optimizer=opt,epoch=ep,state=state)\n    last=tf.train.CheckpointManager(ck,str(out/'last'),max_to_keep=2)\n    best=tf.train.CheckpointManager(snap,str(out/'best'),max_to_keep=None)\n    if latest:ck.restore(latest).assert_existing_objects_matched()\n    info=json.loads(state.numpy().decode())\n    # Stateless dropout makes replay of interrupted epochs deterministic for the same environment.\n    @tf.function\n    def step(x,y,ht,seed):\n        with tf.GradientTape() as tape:\n            hs=fm(prep(x,student),training=True)\n            mask=tf.cast(tf.random.stateless_uniform(tf.shape(hs),seed=seed)>=.2,hs.dtype)\n            hd=hs*mask/.8;cl=m.get_layer('probability')\n            ls=sup(y,tf.matmul(hd,cl.kernel)+cl.bias)\n            lk=fkl(ht,pj(hs)) if teacher!='none' else tf.constant(0.,tf.float32)\n            loss=ls+lam*lk\n            tf.debugging.assert_all_finite(loss,'Nonfinite training loss')\n        grads=tape.gradient(loss,variables)\n        for g in grads:\n            if g is None:raise RuntimeError('Disconnected train gradient')\n            tf.debugging.assert_all_finite(g,'Nonfinite train gradient')\n        opt.apply_gradients(zip(grads,variables));return ls,lk\n    f=FRAMES['train']\n    for epoch in range(int(ep.numpy()),budget):\n        start=time.perf_counter();order=np.random.default_rng(42+epoch).permutation(len(f));tot=np.zeros(2);n=0\n        # Both models receive the same per-image flip; cached features follow original row indices.\n        for batch,start_ix in enumerate(range(0,len(f),8)):\n            ix=order[start_ix:start_ix+8];x=raw_at(f.iloc[ix])\n            flips=tf.random.stateless_uniform([len(ix)],seed=[42+epoch*1009,batch])<.5\n            x=tf.where(flips[:,None,None,None],tf.reverse(x,[2]),x)\n            ht=tf.constant(np.asarray(features[ix,flips.numpy().astype(int),:])) if features is not None else tf.zeros([len(ix),1])\n            ls,lk=step(x,tf.constant(f.iloc[ix].label.to_numpy(),tf.float32),ht,tf.constant([42+epoch*1009+(0 if ishead else 100000),batch]))\n            tot+=np.array([float(ls),float(lk)])*len(ix);n+=len(ix)\n        val=evaluate(m,student,out,dict(model=student,teacher=teacher,dataset=DATASET,training_seed=42,split='validation',stage=stage,phase_epoch=epoch+1))\n        score=[val['macro_f1'],val['recall']]\n        if tuple(score)>tuple(info['best_score']):\n            info['best_score']=score;info['best_checkpoint']=best.save(checkpoint_number=epoch+1);info['best_epoch']=epoch+1\n        info['history'].append(dict(epoch=epoch+1,stage=stage,sup_loss=tot[0]/n,feature_loss=tot[1]/n,lambda_feature=lam,\n            macro_f1=val['macro_f1'],recall=val['recall'],f1=val['f1'],roc_auc=val['roc_auc'],seconds=time.perf_counter()-start))\n        ep.assign(epoch+1);state.assign(json.dumps(info));last.save(checkpoint_number=epoch+1)\n        pd.DataFrame(info['history']).to_csv(out/'history.csv',index=False)\n        print(student,teacher,stage,epoch+1,'macro-F1',round(val['macro_f1'],4),flush=True)\n    if int(ep.numpy())!=budget:raise RuntimeError('Incomplete phase')\n    snap.restore(info['best_checkpoint']).assert_existing_objects_matched()\n    val=evaluate(m,student,out,dict(model=student,teacher=teacher,dataset=DATASET,training_seed=42,split='validation',stage=stage,phase_epoch=info['best_epoch']))\n    pd.DataFrame(info['history']).to_csv(out/'history.csv',index=False)\n    done=dict(best_checkpoint=info['best_checkpoint'],metrics=val,epochs_completed=budget,\n        minutes=sum(r['seconds'] for r in info['history'])/60)\n    save(donepath,done)\n    del m,fm,pj,step,opt,ck,snap,last,best;keras.backend.clear_session();gc.collect();return done"
engine_path=Path('/content/b06b_engine.py')
engine_path.write_text(ENGINE_SOURCE)
linecache.cache[str(engine_path)]=(len(ENGINE_SOURCE),None,ENGINE_SOURCE.splitlines(True),str(engine_path))
exec(compile(ENGINE_SOURCE,str(engine_path),'exec'),globals())


## Cell 5 — Trainer, resume dan validation
Fase head melatih classifier dan projector, sementara backbone beku. Fine-tuning memuat checkpoint head terbaik, lalu melatih backbone dengan BN tetap beku. Projector tidak dibutuhkan pada inferensi. Control memakai lambda=0.

In [ ]:
print('Trainer loaded from engine in Cell 4')


## Cell 6 — Bekukan konfigurasi, kalibrasi seluruh pasangan, lalu training
Kalibrasi selesai untuk semua pasangan sebelum training dimulai. Test tidak dapat diaktifkan dari notebook ini. Bila runtime terputus, jalankan lagi dari Cell1 dengan konfigurasi yang sama.

In [ ]:
CONFIG['engine_sha256']=hashlib.sha256(ENGINE_SOURCE.encode()).hexdigest()
guarded(OUT/'protocol.json',CONFIG)
(OUT/'b06b_engine.py').write_text(ENGINE_SOURCE)
FEATURES={t:teacher_cache(t) for t in ['vgg16','densenet121']}
PLAN=[(s,t) for t in ['none','vgg16','densenet121'] for s in ['mobilenet_v2','nasnet_mobile']]
CAL={}
for student,teacher in PLAN:
    folder=OUT/(student+'__from__'+teacher)/'seed_42';folder.mkdir(parents=True,exist_ok=True)
    if teacher!='none':CAL[(student,teacher)]=calibrate(student,teacher,FEATURES[teacher],folder)
cal_table=pd.DataFrame([dict(student=s,teacher=t,**c) for (s,t),c in CAL.items()])
cal_table.to_csv(OUT/'calibration_summary.csv',index=False)
display(cal_table[['student','teacher','lambda_feature','clipped','cosine_median','achieved_median_weighted_ratio']])
RESULTS=[]
for student,teacher in PLAN:
    out=OUT/(student+'__from__'+teacher)/'seed_42'
    lam=0. if teacher=='none' else CAL[(student,teacher)]['lambda_feature']
    cfg=dict(CONFIG,student=student,teacher=teacher,lambda_feature=lam,
        calibration_sha256=None if teacher=='none' else sha(out/'calibration.json'))
    guarded(out/'config.json',cfg)
    if (out/'COMPLETE.json').exists():
        done=read_json(out/'COMPLETE.json')
        if done['config_sha256']!=sha(out/'config.json'):raise RuntimeError('Completed config changed')
        RESULTS.append(read_json(out/'validation_metrics.json'));continue
    feat=FEATURES.get(teacher)
    h=phase(student,teacher,feat,lam,out/'head','head')
    ft=phase(student,teacher,feat,lam,out/'finetune','finetune',h['best_checkpoint'])
    use_ft=(ft['metrics']['macro_f1'],ft['metrics']['recall'])>(h['metrics']['macro_f1'],h['metrics']['recall'])
    selected=ft if use_ft else h;stage='finetune' if use_ft else 'head'
    if h['epochs_completed']+ft['epochs_completed']!=30:raise RuntimeError('Incomplete run')
    result=dict(selected['metrics'],method='supervised_control' if teacher=='none' else 'ssid_inspired_feature_kd',
        best_stage=stage,best_epoch=selected['metrics']['phase_epoch']+(5 if use_ft else 0),epochs_completed=30,
        lambda_feature=lam,total_train_minutes=h['minutes']+ft['minutes'])
    shutil.copyfile(out/stage/'validation_predictions.csv',out/'validation_predictions.csv')
    save(out/'validation_metrics.json',result)
    labels=['BI-RADS 1/2','BI-RADS 5/6'] if DATASET=='inbreast' else ['Benign','Malignant']
    fig,ax=plt.subplots(figsize=(5.5,5))
    ConfusionMatrixDisplay(np.array([[result['tn'],result['fp']],[result['fn'],result['tp']]]),display_labels=labels).plot(ax=ax,cmap='Blues',values_format='d',im_kw={'vmin':0,'vmax':int(FRAMES['validation'].label.value_counts().max())})
    ax.set_title(f'{student} <- {teacher} | validation');fig.tight_layout();fig.savefig(out/'validation_confusion_matrix.png',dpi=160);plt.close(fig)
    save(out/'COMPLETE.json',dict(best_checkpoint=selected['best_checkpoint'],best_stage=stage,config_sha256=sha(out/'config.json'),epochs_completed=30,validation_only=True))
    RESULTS.append(result);pd.DataFrame(RESULTS).to_csv(OUT/'validation_summary.csv',index=False)
pd.DataFrame(RESULTS).to_csv(OUT/'validation_summary.csv',index=False)
# Confirm all paired students began from the same weights.
for student in ['mobilenet_v2','nasnet_mobile']:
    hashes=[read_json(OUT/(student+'__from__'+t)/'seed_42/head/initialization.json')['student_sha256'] for t in ['none','vgg16','densenet121']]
    if len(set(hashes))!=1:raise RuntimeError('Unmatched initialization')
    for teacher in ['vgg16','densenet121']:
        if CAL[(student,teacher)]['initial_student_sha256']!=hashes[0]:raise RuntimeError('Calibration initialization differs')
print('Selesai enam run validation. Test tetap tertutup.')

## Cell 7 — Ringkasan, delta control, dan ZIP
Waktu training tidak termasuk cache Teacher dan kalibrasi; keduanya dilaporkan terpisah. Simpan hasil satu seed sebagai deskriptif. Pada kegagalan Cell6, Cell7 masih bisa mengemas hasil parsial.

In [ ]:
records=[read_json(p) for p in OUT.glob('*/seed_42/validation_metrics.json') if (p.parent/'COMPLETE.json').exists()]
if records:
    df=pd.DataFrame(records);df.to_csv(OUT/'validation_summary.csv',index=False)
    display(df[['model','teacher','precision','recall','f1','macro_f1','specificity','roc_auc','lambda_feature']].round(4))
    rows=[]
    for r in records:
        if r['teacher']=='none':continue
        controls=[b for b in records if b['model']==r['model'] and b['teacher']=='none']
        if len(controls)==1:rows.append(dict(model=r['model'],teacher=r['teacher'],**{'delta_'+k:r[k]-controls[0][k] for k in ['precision','recall','f1','macro_f1','specificity','accuracy','roc_auc']}))
    pd.DataFrame(rows).to_csv(OUT/'validation_kd_vs_control.csv',index=False)
report=OUT/f'B06B_{DATASET}_seed42_validation_report.zip'
with zipfile.ZipFile(report,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and p.suffix in ['.json','.csv','.png','.py']:z.write(p,str(p.relative_to(OUT)))
print('Completed runs:',len(records),'/6');print(report)
if DOWNLOAD_REPORT:files.download(str(report))